# Phase 4: Fine-Tuning with LoRA/PEFT

**Goal:** Understand how to adapt a pre-trained model to your specific task by modifying its weights.

You now know:
- How models work internally (Phase 1)
- How they generate text (Phase 2)
- How prompts steer output without changing the model (Phase 3)

**This phase:** When prompting isn't enough — actually change the model to specialize it.

We'll fine-tune **FLAN-T5-small** on dialogue summarization and compare:
1. Base model (no fine-tuning)
2. Full fine-tuning (update all weights)
3. LoRA/PEFT (update only a tiny fraction of weights)

In [ ]:
import torch
import numpy as np
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
)
from datasets import load_dataset
import evaluate
from peft import LoraConfig, get_peft_model, TaskType
import time

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Device: {device}")
print(f"Using FLAN-T5-small (60M params) for faster training on CPU/MPS")

---
## 1. Setup: Load Data and Model

We use a small subset of DialogSum to keep training fast on your laptop.
In production, you'd use thousands of examples — here we use ~200 for learning.

In [ ]:
# Load dataset — small subset for fast training
dataset = load_dataset("knkarthick/dialogsum")

train_data = dataset["train"].select(range(200))
val_data = dataset["validation"].select(range(50))
test_data = dataset["test"].select(range(50))

print(f"Train: {len(train_data)} examples")
print(f"Val:   {len(val_data)} examples")
print(f"Test:  {len(test_data)} examples")
print(f"\nExample:")
print(f"  Dialogue: {train_data[0]['dialogue'][:100]}...")
print(f"  Summary:  {train_data[0]['summary']}")

In [ ]:
# Load model and tokenizer
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess(examples):
    """Tokenize dialogues and summaries for training."""
    prefix = "Summarize the following dialogue:\n\n"
    inputs = [prefix + d for d in examples["dialogue"]]
    targets = examples["summary"]
    
    model_inputs = tokenizer(
        inputs, max_length=512, truncation=True, padding="max_length"
    )
    labels = tokenizer(
        targets, max_length=128, truncation=True, padding="max_length"
    )
    # Replace padding token ids with -100 so they're ignored in loss
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Preprocess datasets
train_tokenized = train_data.map(preprocess, batched=True, remove_columns=train_data.column_names)
val_tokenized = val_data.map(preprocess, batched=True, remove_columns=val_data.column_names)
test_tokenized = test_data.map(preprocess, batched=True, remove_columns=test_data.column_names)

print(f"Tokenized train sample keys: {list(train_tokenized[0].keys())}")
print(f"Input length: {len(train_tokenized[0]['input_ids'])} tokens")

In [ ]:
# Evaluation helper — generates summaries and computes ROUGE

rouge = evaluate.load("rouge")

def evaluate_model(model, test_data, tokenizer, num_samples=50):
    """Generate summaries and compute ROUGE scores."""
    model.eval()
    predictions = []
    references = []
    
    for i in range(min(num_samples, len(test_data))):
        dialogue = test_data[i]["dialogue"]
        reference = test_data[i]["summary"]
        
        prompt = f"Summarize the following dialogue:\n\n{dialogue}"
        inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=128)
        
        pred = tokenizer.decode(output[0], skip_special_tokens=True)
        predictions.append(pred)
        references.append(reference)
    
    scores = rouge.compute(predictions=predictions, references=references)
    return scores, predictions, references

---
## 2. Baseline: How Good Is the Model Before Fine-Tuning?

In [ ]:
# Load fresh model for baseline
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

total_params = sum(p.numel() for p in base_model.parameters())
print(f"Model: {model_name}")
print(f"Total parameters: {total_params:,}")

print(f"\nEvaluating baseline (no fine-tuning)...")
baseline_scores, baseline_preds, baseline_refs = evaluate_model(
    base_model, test_data, tokenizer
)

print(f"\nBaseline ROUGE scores:")
print(f"  ROUGE-1: {baseline_scores['rouge1']:.4f}")
print(f"  ROUGE-2: {baseline_scores['rouge2']:.4f}")
print(f"  ROUGE-L: {baseline_scores['rougeL']:.4f}")

print(f"\nSample predictions:")
for i in range(3):
    print(f"\n  Reference:  {baseline_refs[i]}")
    print(f"  Predicted:  {baseline_preds[i]}")

---
## 3. Full Fine-Tuning: Update All Parameters

The simplest approach: train the entire model on your dataset. Every weight gets updated.

**Pros:** Maximum adaptation to your task
**Cons:** Slow, memory-intensive, risk of catastrophic forgetting

In [ ]:
# Full fine-tuning
full_ft_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

trainable_params = sum(p.numel() for p in full_ft_model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable_params:,} (100% of model)")

training_args = Seq2SeqTrainingArguments(
    output_dir="../outputs/full-ft",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=3e-4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    predict_with_generate=True,
    fp16=False,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=full_ft_model)

trainer = Seq2SeqTrainer(
    model=full_ft_model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("Starting full fine-tuning (this may take 5-10 minutes on CPU)...")
start_time = time.time()
trainer.train()
full_ft_time = time.time() - start_time
print(f"\nFull fine-tuning completed in {full_ft_time:.1f} seconds")

In [ ]:
# Evaluate full fine-tuned model
print("Evaluating full fine-tuned model...")
full_ft_scores, full_ft_preds, full_ft_refs = evaluate_model(
    full_ft_model, test_data, tokenizer
)

print(f"\nFull Fine-Tuned ROUGE scores:")
print(f"  ROUGE-1: {full_ft_scores['rouge1']:.4f}  (baseline: {baseline_scores['rouge1']:.4f})")
print(f"  ROUGE-2: {full_ft_scores['rouge2']:.4f}  (baseline: {baseline_scores['rouge2']:.4f})")
print(f"  ROUGE-L: {full_ft_scores['rougeL']:.4f}  (baseline: {baseline_scores['rougeL']:.4f})")

print(f"\nSample predictions (full fine-tuned):")
for i in range(3):
    print(f"\n  Reference:  {full_ft_refs[i]}")
    print(f"  Baseline:   {baseline_preds[i]}")
    print(f"  Fine-tuned: {full_ft_preds[i]}")

### YOUR UNDERSTANDING

*Did fine-tuning improve the ROUGE scores? By how much?*

> (your notes here)


---
## 4. The Problem: Catastrophic Forgetting

Full fine-tuning changes **every** weight. The model gets better at your specific task but can **forget** how to do other things it knew before.

In [ ]:
# Test catastrophic forgetting — can the model still do general tasks?

general_tasks = [
    "Translate to French: The cat is sleeping.",
    "What is 2 + 2?",
    "Is the following sentence positive or negative? I love this product!",
]

base_check = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("General task performance: Base vs Full Fine-Tuned\n")
for task in general_tasks:
    # Base model
    inputs = tokenizer(task, return_tensors="pt", max_length=512, truncation=True)
    with torch.no_grad():
        base_out = base_check.generate(**inputs, max_new_tokens=50)
        ft_out = full_ft_model.generate(
            **{k: v.to(full_ft_model.device) for k, v in inputs.items()},
            max_new_tokens=50
        )
    
    base_text = tokenizer.decode(base_out[0], skip_special_tokens=True)
    ft_text = tokenizer.decode(ft_out[0], skip_special_tokens=True)
    
    print(f"Task: {task}")
    print(f"  Base model:  {base_text}")
    print(f"  Fine-tuned:  {ft_text}")
    print()

del base_check

print("If the fine-tuned model gives worse or nonsensical answers on general tasks,")
print("that's catastrophic forgetting — it specialized too hard on summarization.")
print("\nThis is the main motivation for PEFT/LoRA (next section).")

### YOUR UNDERSTANDING

*What is catastrophic forgetting and why does it happen?*

> (your notes here)


---
## 5. LoRA: The Efficient Alternative

**LoRA (Low-Rank Adaptation)** is the key idea behind PEFT:
- Freeze the original model weights entirely
- Add tiny trainable matrices alongside specific layers
- Only train these small additions (~1-5% of total parameters)

**Why it works:** Most of what a pre-trained model knows is already good. You only need small adjustments for your specific task.

```
Full fine-tuning:    Update ALL 60M parameters
LoRA:                Freeze 60M, train ~300K new parameters (0.5%)
```

In [ ]:
# Set up LoRA
lora_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

lora_config = LoraConfig(
    r=8,                           # rank of the update matrices (lower = fewer params)
    lora_alpha=32,                 # scaling factor
    target_modules=["q", "v"],     # which layers to add LoRA to
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,
)

lora_model = get_peft_model(lora_model, lora_config)

# Compare parameter counts
total = sum(p.numel() for p in lora_model.parameters())
trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
frozen = total - trainable

print(f"LoRA Configuration:")
print(f"  Rank (r): {lora_config.r}")
print(f"  Target modules: {lora_config.target_modules}")
print(f"\nParameter comparison:")
print(f"  Total parameters:     {total:>12,}")
print(f"  Frozen parameters:    {frozen:>12,}  ({frozen/total:.1%})")
print(f"  Trainable (LoRA):     {trainable:>12,}  ({trainable/total:.1%})")
print(f"\n  Full fine-tuning trains {total:,} parameters")
print(f"  LoRA trains only {trainable:,} parameters — {total/trainable:.0f}x fewer!")

In [ ]:
# Visualize where LoRA adds parameters
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart: trainable vs frozen
axes[0].pie(
    [frozen, trainable],
    labels=[f"Frozen\n{frozen:,}", f"LoRA trainable\n{trainable:,}"],
    colors=["#bdc3c7", "#e74c3c"],
    autopct="%1.1f%%",
    startangle=90,
    textprops={"fontsize": 10},
)
axes[0].set_title("LoRA: Trainable vs Frozen Parameters")

# Bar chart: full FT vs LoRA
methods = ["Full Fine-Tuning", "LoRA"]
params = [total, trainable]
colors = ["#3498db", "#e74c3c"]
bars = axes[1].bar(methods, params, color=colors, edgecolor="black")
axes[1].set_ylabel("Trainable Parameters")
axes[1].set_title("Parameters to Train")
for bar, p in zip(bars, params):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                f"{p:,}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Train with LoRA
lora_training_args = Seq2SeqTrainingArguments(
    output_dir="../outputs/lora-ft",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=3e-4,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    predict_with_generate=True,
    fp16=False,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

lora_data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=lora_model)

lora_trainer = Seq2SeqTrainer(
    model=lora_model,
    args=lora_training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=lora_data_collator,
)

print(f"Starting LoRA fine-tuning (training only {trainable:,} parameters)...")
start_time = time.time()
lora_trainer.train()
lora_ft_time = time.time() - start_time
print(f"\nLoRA fine-tuning completed in {lora_ft_time:.1f} seconds")
print(f"Full fine-tuning took: {full_ft_time:.1f} seconds")
print(f"Speedup: {full_ft_time / lora_ft_time:.1f}x faster")

In [ ]:
# Evaluate LoRA model
print("Evaluating LoRA fine-tuned model...")
lora_scores, lora_preds, lora_refs = evaluate_model(
    lora_model, test_data, tokenizer
)

print(f"\n{'Method':<25} {'ROUGE-1':<12} {'ROUGE-2':<12} {'ROUGE-L':<12} {'Train Time'}")
print("─" * 75)
print(f"{'Baseline (no FT)':<25} {baseline_scores['rouge1']:<12.4f} {baseline_scores['rouge2']:<12.4f} {baseline_scores['rougeL']:<12.4f} {'—'}")
print(f"{'Full Fine-Tuning':<25} {full_ft_scores['rouge1']:<12.4f} {full_ft_scores['rouge2']:<12.4f} {full_ft_scores['rougeL']:<12.4f} {full_ft_time:.0f}s")
print(f"{'LoRA (r=8)':<25} {lora_scores['rouge1']:<12.4f} {lora_scores['rouge2']:<12.4f} {lora_scores['rougeL']:<12.4f} {lora_ft_time:.0f}s")

print(f"\nSample predictions comparison:")
for i in range(3):
    print(f"\n  Reference:    {lora_refs[i]}")
    print(f"  Baseline:     {baseline_preds[i]}")
    print(f"  Full FT:      {full_ft_preds[i]}")
    print(f"  LoRA:         {lora_preds[i]}")

### YOUR UNDERSTANDING

*How does LoRA's performance compare to full fine-tuning? Is the trade-off worth it?*

> (your notes here)


---
## 6. LoRA Hyperparameters: What Can You Tune?

LoRA's key parameter is **rank (r)** — it controls how many new parameters you add:
- Lower r = fewer parameters, faster training, more compression
- Higher r = more parameters, better adaptation, closer to full fine-tuning

In [ ]:
# Compare different LoRA ranks

ranks = [2, 4, 8, 16]

print(f"{'Rank':<8} {'Trainable Params':<20} {'% of Total':<15}")
print("─" * 45)

for r in ranks:
    temp_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    config = LoraConfig(
        r=r, lora_alpha=32, target_modules=["q", "v"],
        lora_dropout=0.05, bias="none", task_type=TaskType.SEQ_2_SEQ_LM,
    )
    temp_model = get_peft_model(temp_model, config)
    trainable = sum(p.numel() for p in temp_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in temp_model.parameters())
    print(f"r={r:<5} {trainable:>12,}       {trainable/total:.2%}")
    del temp_model

print(f"\nRule of thumb:")
print(f"  r=4-8 is usually sufficient for most tasks")
print(f"  r=16+ only needed for very complex domain adaptation")
print(f"  Higher r approaches full fine-tuning in cost")

---
## 7. Catastrophic Forgetting: LoRA vs Full Fine-Tuning

In [ ]:
# Does LoRA preserve general capabilities better?

general_tasks = [
    "Translate to French: The cat is sleeping.",
    "What is 2 + 2?",
    "Is the following sentence positive or negative? I love this product!",
]

base_fresh = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("General task performance: Base vs Full FT vs LoRA\n")
print(f"{'Task':<55} {'Base':<20} {'Full FT':<20} {'LoRA'}")
print("─" * 120)

for task in general_tasks:
    inputs = tokenizer(task, return_tensors="pt", max_length=512, truncation=True)
    
    with torch.no_grad():
        base_out = base_fresh.generate(**inputs, max_new_tokens=50)
        ft_out = full_ft_model.generate(
            **{k: v.to(full_ft_model.device) for k, v in inputs.items()},
            max_new_tokens=50
        )
        lora_out = lora_model.generate(
            **{k: v.to(lora_model.device) for k, v in inputs.items()},
            max_new_tokens=50
        )
    
    base_text = tokenizer.decode(base_out[0], skip_special_tokens=True)
    ft_text = tokenizer.decode(ft_out[0], skip_special_tokens=True)
    lora_text = tokenizer.decode(lora_out[0], skip_special_tokens=True)
    
    task_short = task[:52] + "..." if len(task) > 55 else task
    print(f"{task_short:<55} {base_text:<20} {ft_text:<20} {lora_text}")

del base_fresh

print(f"\nLoRA should preserve general capabilities better because the original")
print(f"weights are frozen — only the small LoRA adapters are changed.")

---
## 8. Summary: When to Use What

| Approach | Params trained | Speed | Forgetting risk | When to use |
|---|---|---|---|---|
| **Prompt engineering** | 0 | Instant | None | First thing to try |
| **LoRA/PEFT** | ~1% | Fast | Low | Task-specific adaptation |
| **Full fine-tuning** | 100% | Slow | High | Deep domain specialization |

### The Practical Hierarchy
```
1. Try prompt engineering first (free, instant)
      ↓ not good enough?
2. Try LoRA/PEFT (cheap, fast, minimal forgetting)
      ↓ still not good enough?
3. Full fine-tuning (expensive, slow, but maximum adaptation)
      ↓ still not good enough?
4. Train a custom model from scratch (rarely needed)
```

### YOUR UNDERSTANDING — Final Reflection

*What is LoRA and why is it preferred over full fine-tuning in most cases?*

> (your notes here)

*If you were fine-tuning a model for customer support ticket summarization, which approach would you start with and why?*

> (your notes here)

*What was the most surprising result in this notebook?*

> (your notes here)


---
## Next Up: Phase 5 — Retrieval Augmented Generation (RAG)

Fine-tuning changes the model to know your domain better. But what if the information changes frequently (new products, updated policies)? You can't retrain every time.

RAG solves this by keeping the model fixed and feeding it relevant information from a database at query time. This is the core technique behind the Customer Support project.

**Before moving on**, fill in all `YOUR UNDERSTANDING` sections!